In [ ]:
import os, time, math, heapq, statistics
from collections import deque
import osmnx as ox
import networkx as nx
import requests
import pandas as pd
import folium
from folium.plugins import LayerControl
from dotenv import load_dotenv

load_dotenv()
TOMTOM_KEY = os.environ["TOMTOM_API_KEY"]

CENTER      = (51.5080, -0.1281)
DESTINATION = (51.4988, -0.1749)
DIST_M      = 2500

ORIGINS = [
    ("Trafalgar",    (51.5080, -0.1281)),
    ("KingsCross",   (51.5308, -0.1238)),
    ("Waterloo",     (51.5033, -0.1147)),
]

V_MAX_KPH = 130.0

In [ ]:
G = ox.graph_from_point(CENTER, dist=DIST_M, network_type="drive", simplify=True)
G = nx.MultiDiGraph(G)
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print(f"Nodes: {len(G.nodes):,}   Edges: {len(G.edges):,}")
print(f"CRS: {G.graph['crs']}")

In [ ]:
FLOW_URL = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

def tomtom_flow(lat, lon):
    try:
        r = requests.get(
            FLOW_URL,
            params={"key": TOMTOM_KEY, "point": f"{lat},{lon}",
                    "unit": "KMPH", "thickness": 1},
            timeout=10,
        )
        if r.status_code != 200:
            return None
        return r.json().get("flowSegmentData")
    except Exception:
        return None

def edge_midpoint(u, v, k):
    d = G.get_edge_data(u, v, k)
    geom = d.get("geometry")
    if geom is None:
        yu, xu = G.nodes[u]["y"], G.nodes[u]["x"]
        yv, xv = G.nodes[v]["y"], G.nodes[v]["x"]
        return ((yu + yv) / 2, (xu + xv) / 2)
    coords = list(geom.coords) if hasattr(geom, "coords") else list(geom)
    mid = coords[len(coords) // 2]
    return (mid[1], mid[0])

def bearing(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    x = math.sin(dl) * math.cos(p2)
    y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def bearing_diff(b1, b2):
    d = abs(b1 - b2) % 360
    return min(d, 360 - d)

SAMPLE_EVERY = max(1, len(G.edges) // 500)

traffic = {}
n_queried = n_matched = n_closed = n_missing = 0

for i, (u, v, k) in enumerate(G.edges(keys=True)):
    if i % SAMPLE_EVERY:
        continue
    mid_lat, mid_lon = edge_midpoint(u, v, k)
    flow = tomtom_flow(mid_lat, mid_lon)
    n_queried += 1
    if flow is None:
        n_missing += 1
        continue
    if flow.get("roadClosure", False):
        n_closed += 1
        traffic[(u, v, k)] = {"closure": True}
        continue

    v_cur  = flow.get("currentSpeed", 0)
    v_free = flow.get("freeFlowSpeed", 0)
    conf   = flow.get("confidence", 1.0)

    coord_block = flow.get("coordinates")
    ft = coord_block.get("coordinate", []) if isinstance(coord_block, dict) else []
    if ft and len(ft) >= 2:
        tv_b = bearing(ft[0]["latitude"], ft[0]["longitude"],
                       ft[-1]["latitude"], ft[-1]["longitude"])
        osm_b = bearing(G.nodes[u]["y"], G.nodes[u]["x"],
                        G.nodes[v]["y"], G.nodes[v]["x"])
        if bearing_diff(tv_b, osm_b) > 90:
            n_missing += 1
            continue

    if v_cur <= 0 or v_free <= 0 or conf < 0.5:
        n_missing += 1
        continue

    traffic[(u, v, k)] = {
        "closure": False,
        "v_current": v_cur,
        "v_free": v_free,
        "confidence": conf,
        "ratio": v_cur / v_free,
    }
    n_matched += 1

print(f"Queried: {n_queried}   Matched: {n_matched}   "
      f"Closures: {n_closed}   Missing/low-confidence: {n_missing}")

for (u, v, k) in G.edges(keys=True):
    d = G.get_edge_data(u, v, k)
    if (u, v, k) in traffic:
        continue
    spd = d.get("speed_kph", 30) or 30
    traffic[(u, v, k)] = {
        "closure": False,
        "v_current": spd,
        "v_free": spd,
        "confidence": 0.0,
        "ratio": 1.0,
        "imputed": True,
    }

for (u, v, k), tr in traffic.items():
    d = G.get_edge_data(u, v, k)
    if tr.get("closure"):
        d["t_cost"] = float("inf")
    else:
        d["t_cost"] = 3.6 * d["length"] / tr["v_current"]
    d["d_cost"] = d["length"]
    d["traffic"] = tr

In [ ]:
def _haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def h_distance(node, goal_node, G):
    a, b = G.nodes[node], G.nodes[goal_node]
    return _haversine_m(a["y"], a["x"], b["y"], b["x"])

def h_time(node, goal_node, G):
    return 3.6 * h_distance(node, goal_node, G) / V_MAX_KPH

def _neighbors(G, u):
    if u not in G:
        return
    for v, k, d in sorted(G.out_edges(u, keys=True, data=True),
                          key=lambda t: (t[0], t[1])):
        yield v, k, d

def _reconstruct(came, goal):
    path = []
    u = goal
    while u is not None:
        path.append(u)
        u = came[u]
    return list(reversed(path))

def bfs(G, start, goal, objective="d_cost"):
    frontier = deque([start])
    came = {start: None}
    explored = set()
    expansions = 0
    max_frontier = 1
    while frontier:
        max_frontier = max(max_frontier, len(frontier))
        u = frontier.popleft()
        if u == goal:
            return _reconstruct(came, goal), expansions, max_frontier
        explored.add(u)
        expansions += 1
        for v, _, _ in _neighbors(G, u):
            if v not in explored and v not in came:
                came[v] = u
                frontier.append(v)
    return None, expansions, max_frontier

def dfs(G, start, goal, objective="d_cost"):
    frontier = [start]
    came = {start: None}
    explored = set()
    expansions = 0
    max_frontier = 1
    while frontier:
        max_frontier = max(max_frontier, len(frontier))
        u = frontier.pop()
        if u == goal:
            return _reconstruct(came, goal), expansions, max_frontier
        if u in explored:
            continue
        explored.add(u)
        expansions += 1
        nbrs = [v for v, _, _ in _neighbors(G, u)]
        for v in reversed(nbrs):
            if v not in explored and v not in came:
                came[v] = u
                frontier.append(v)
    return None, expansions, max_frontier

def greedy_bfs(G, start, goal, objective="d_cost"):
    h = h_time if objective == "t_cost" else h_distance
    heap = [(h(start, goal, G), start)]
    came = {start: None}
    explored = set()
    expansions = 0
    max_frontier = 1
    while heap:
        max_frontier = max(max_frontier, len(heap))
        _, u = heapq.heappop(heap)
        if u == goal:
            return _reconstruct(came, goal), expansions, max_frontier
        if u in explored:
            continue
        explored.add(u)
        expansions += 1
        for v, _, _ in _neighbors(G, u):
            if v not in explored and v not in came:
                came[v] = u
                heapq.heappush(heap, (h(v, goal, G), v))
    return None, expansions, max_frontier

def astar(G, start, goal, objective="d_cost"):
    h = h_time if objective == "t_cost" else h_distance
    heap = [(h(start, goal, G), 0.0, start)]
    best_g = {start: 0.0}
    came = {start: None}
    explored = set()
    expansions = 0
    max_frontier = 1
    while heap:
        max_frontier = max(max_frontier, len(heap))
        f, g, u = heapq.heappop(heap)
        if u == goal:
            return _reconstruct(came, goal), expansions, max_frontier
        if u in explored:
            continue
        explored.add(u)
        expansions += 1
        for v, _, d in _neighbors(G, u):
            w = d.get(objective, float("inf"))
            if w == float("inf"):
                continue
            ng = g + w
            if ng < best_g.get(v, float("inf")):
                best_g[v] = ng
                came[v] = u
                heapq.heappush(heap, (ng + h(v, goal, G), ng, v))
    return None, expansions, max_frontier

In [ ]:
def path_metrics(G, path, objective="d_cost"):
    if not path:
        return {"distance_m": None, "time_s": None, "n_edges": 0}
    d = t = 0.0
    n = 0
    for a, b in zip(path[:-1], path[1:]):
        candidates = G.get_edge_data(a, b)
        best = min(candidates.values(), key=lambda e: e.get(objective, float("inf")))
        d += best.get("length", 0)
        t += best.get("t_cost", 0)
        n += 1
    return {"distance_m": d, "time_s": t, "n_edges": n}

def snap_to_graph(point, G):
    return ox.distance.nearest_nodes(G, X=point[1], Y=point[0])

goal_node = snap_to_graph(DESTINATION, G)
algos = {"BFS": bfs, "DFS": dfs, "Greedy": greedy_bfs, "A*": astar}

records = []
for origin_name, origin_pt in ORIGINS:
    start_node = snap_to_graph(origin_pt, G)
    for objective in ("d_cost", "t_cost"):
        for algo_name, fn in algos.items():
            runtimes = []
            last = None
            for _ in range(5):
                t0 = time.perf_counter()
                path, ex, mf = fn(G, start_node, goal_node, objective=objective)
                runtimes.append(time.perf_counter() - t0)
                last = (path, ex, mf)
            path, ex, mf = last
            m = path_metrics(G, path, objective)
            records.append({
                "origin": origin_name, "objective": objective, "algo": algo_name,
                "runtime_med_s": statistics.median(runtimes),
                "expansions": ex, "max_frontier": mf,
                "distance_m": m["distance_m"], "time_s": m["time_s"],
                "n_edges": m["n_edges"], "path": path,
            })

df = pd.DataFrame([{k: v for k, v in r.items() if k != "path"} for r in records])
print(df.to_string(index=False))

In [ ]:
colors = {"BFS": "red", "DFS": "orange", "Greedy": "purple", "A*": "blue"}
m = folium.Map(location=CENTER, zoom_start=14, tiles="cartodbpositron")

folium.Marker(DESTINATION, tooltip="Destination",
              icon=folium.Icon(color="green")).add_to(m)

for origin_name, origin_pt in ORIGINS:
    folium.Marker(origin_pt, tooltip=origin_name,
                  icon=folium.Icon(color="black")).add_to(m)

for r in records:
    if not r["path"]:
        continue
    coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in r["path"]]
    fg = folium.FeatureGroup(name=f"{r['origin']} | {r['objective']} | {r['algo']}")
    folium.PolyLine(coords, color=colors[r["algo"]], weight=4, opacity=0.8,
                    tooltip=f"{r['algo']} {r['objective']}").add_to(fg)
    fg.add_to(m)

LayerControl(collapsed=False).add_to(m)
m

In [ ]:
overlay = folium.Map(location=CENTER, zoom_start=14, tiles="cartodbpositron")

for (u, v, k), tr in traffic.items():
    if tr.get("closure"):
        color, weight = "black", 4
    else:
        r_ = tr.get("ratio", 1.0)
        if r_ < 0.5:
            color, weight = "red", 3
        elif r_ < 0.8:
            color, weight = "yellow", 3
        else:
            continue
    y1, x1 = G.nodes[u]["y"], G.nodes[u]["x"]
    y2, x2 = G.nodes[v]["y"], G.nodes[v]["x"]
    folium.PolyLine([(y1, x1), (y2, x2)], color=color,
                    weight=weight, opacity=0.7).add_to(overlay)

overlay

In [ ]:
def pick_closure_edge(G, start_node, goal_node):
    try:
        base_path, _, _ = astar(G, start_node, goal_node, objective="t_cost")
    except Exception:
        return None
    if not base_path or len(base_path) < 3:
        return None
    mid = base_path[len(base_path) // 2]
    candidates = list(G.out_edges(mid, keys=True))
    return candidates[0] if candidates else None

closure_records = []
for origin_name, origin_pt in ORIGINS:
    start_node = snap_to_graph(origin_pt, G)
    edge = pick_closure_edge(G, start_node, goal_node)
    if edge is None:
        continue
    u0, v0, k0 = edge
    old = G[u0][v0][k0]["t_cost"]
    G[u0][v0][k0]["t_cost"] = float("inf")

    for algo_name, fn in algos.items():
        runtimes = []
        last = None
        for _ in range(5):
            t0 = time.perf_counter()
            path, ex, mf = fn(G, start_node, goal_node, objective="t_cost")
            runtimes.append(time.perf_counter() - t0)
            last = (path, ex, mf)
        path, ex, mf = last
        m = path_metrics(G, path, "t_cost")
        closure_records.append({
            "origin": origin_name, "algo": algo_name,
            "runtime_med_s": statistics.median(runtimes),
            "expansions": ex, "max_frontier": mf,
            "distance_m": m["distance_m"], "time_s": m["time_s"],
        })

    G[u0][v0][k0]["t_cost"] = old

df_closure = pd.DataFrame(closure_records)
print(df_closure.to_string(index=False))

In [ ]:
max_violation = 0.0
for u, v, k, d in G.edges(keys=True, data=True):
    if d.get("t_cost", float("inf")) == float("inf"):
        continue
    h_u = h_time(u, goal_node, G)
    h_v = h_time(v, goal_node, G)
    slack = d["t_cost"] + h_v - h_u
    max_violation = min(max_violation, slack)

print(f"max consistency violation (should be <= 0): {max_violation:.6f}")